# GPU Performance Prediction — Rating Standard Model (std.ipynb)

## 核心挑战：域外推
- 训练集：T4, L4, L40S, V100, A30（低端+中端）
- 测试集：A100, H100, H200 **从未在训练中出现**（83% 测试样本）

## 策略
1. **物理代理特征**：从 transistor / clock / memory_type / architecture 估算算力和带宽
2. **5 模型 Stacking**：LGB + XGB + RF + **KRR** + **CatBoost** → Ridge 元学习器
3. **KRR**：RBF 核捕捉连续特征的非线性（物理代理特征是连续值，核方法擅长）
4. **CatBoost**：原生处理类别特征 + Ordered Boosting 减少过拟合
5. **逐目标**：latency/throughput 做 log 变换；power/efficiency 用原始值

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.kernel_ridge import KernelRidge

import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

RANDOM_SEED = 42
print('Libraries imported.')

Libraries imported.


In [2]:
df_all = pd.read_csv('train.csv')
df_a = pd.read_csv('A.csv')
df_b = pd.read_csv('B.csv')

TARGET_COLS = [
    'gpu_power_draw_watts', 'avg_e2e_latency_seconds',
    'energy_efficiency_tokens_per_joule', 'throughput_tokens_per_second',
]

train_gpus = set(df_all['gpu_type'].unique())
test_gpus  = set(df_a['gpu_type'].unique()) | set(df_b['gpu_type'].unique())
unseen = test_gpus - train_gpus
print(f'Train: {len(df_all)} rows | GPUs: {sorted(train_gpus)}')
print(f'Test:  {len(df_a)+len(df_b)} rows')
print(f'UNSEEN in train: {sorted(unseen)}')

Train: 555 rows | GPUs: ['NVIDIA A30', 'NVIDIA L40S', 'Tesla V100-SXM2-32GB']
Test:  2923 rows
UNSEEN in train: ['NVIDIA A100-SXM4-40GB', 'NVIDIA H100 NVL', 'NVIDIA H200']


## 1. 物理代理特征工程

In [3]:
def engineer_features(df):
    """Feature engineering with public GPU spec lookup + physics proxies."""
    df = df.copy()

    # ── Public GPU specifications (from manufacturer datasheets) ──
    # Contestants CAN and SHOULD look up these specs — they are public knowledge.
    # This lookup provides PRECISE bandwidth/TFLOPS/TDP for EXACT roofline calculations,
    # which is the key to OOD extrapolation (especially to unseen GPUs).
    GPU_LOOKUP = {
        'Tesla T4':                {'bw': 320,  'tflops': 65,  'tdp': 70},
        'NVIDIA L4':               {'bw': 300,  'tflops': 121, 'tdp': 72},
        'NVIDIA L40S':             {'bw': 864,  'tflops': 362, 'tdp': 350},
        'Tesla V100-SXM2-32GB':    {'bw': 900,  'tflops': 125, 'tdp': 300},
        'NVIDIA A30':              {'bw': 933,  'tflops': 165, 'tdp': 165},
        'NVIDIA A100-SXM4-40GB':   {'bw': 1555, 'tflops': 312, 'tdp': 400},
        'NVIDIA H100 NVL':         {'bw': 3350, 'tflops': 835, 'tdp': 450},
        'NVIDIA H200':             {'bw': 4800, 'tflops': 835, 'tdp': 700},
    }
    if 'gpu_type' in df.columns:
        for spec in ['bw', 'tflops', 'tdp']:
            df[f'gpu_{spec}_lookup'] = df['gpu_type'].map(
                lambda g, s=spec: GPU_LOOKUP.get(g, {}).get(s, 0)
            )
        # Roofline features using PRECISE bandwidth/TFLOPS
        bw = df['gpu_bw_lookup'].clip(lower=0.01)
        tf = df['gpu_tflops_lookup'].clip(lower=0.01)
        if 'total_b_params' in df.columns:
            B = df['total_b_params']
            # Memory-bound latency (seconds): time to read all FP16 weights
            df['roofline_mem_latency'] = 2.0 * B / bw
            # Compute-bound latency (seconds): 2 FLOP/param/token / TFLOPS
            df['roofline_comp_latency'] = 2.0 * B * 1e9 / (tf * 1e12)
            # Bandwidth utilization proxy
            df['model_bw_ratio_exact'] = B / bw
            df['model_tf_ratio_exact'] = B / tf
        # GPU spec ratios
        df['bw_per_tflops'] = bw / tf.clip(lower=0.01)

    # ── Architecture-based compute proxy (for comparison / robustness) ──
    arch_gen_map = {'Turing': 1, 'Ampere': 2, 'Ada Lovelace': 3, 'Hopper': 4}
    if 'architecture' in df.columns:
        df['arch_gen'] = df['architecture'].map(arch_gen_map).fillna(2)

    if 'transistor_count_m' in df.columns and 'boost_clock_mhz' in df.columns:
        tc = df['transistor_count_m'].clip(lower=1)
        bc = df['boost_clock_mhz'].clip(lower=100)
        df['compute_proxy'] = tc * bc * bc / 1e9
        df['log_transistor'] = np.log(tc)
        df['log_compute'] = np.log(df['compute_proxy'].clip(lower=0.01))

    # ── Clock features ──
    if 'boost_clock_mhz' in df.columns and 'base_clock_mhz' in df.columns:
        df['clock_ratio'] = df['boost_clock_mhz'] / df['base_clock_mhz'].clip(lower=1)

    # ── Model architecture features ──
    if 'total_b_params' in df.columns:
        df['log_params'] = np.log(df['total_b_params'].clip(lower=0.001))
    if 'hidden_size' in df.columns and 'num_layers' in df.columns:
        df['hidden_per_layer'] = df['hidden_size'] / df['num_layers'].clip(lower=1)
    if 'num_attention_heads' in df.columns and 'num_key_value_heads' in df.columns:
        df['gqa_ratio'] = df['num_attention_heads'] / df['num_key_value_heads'].clip(lower=1)

    # ── Workload features ──
    if 'avg_prompt_tokens' in df.columns and 'avg_generation_tokens' in df.columns:
        df['total_tokens'] = df['avg_prompt_tokens'] + df['avg_generation_tokens']
    if 'lambda_qps' in df.columns:
        df['is_offline'] = (df['lambda_qps'] < 0).astype(int)

    return df

df_fe = engineer_features(df_all)

# ── Target encoding: model_type mean per target (transferable across GPUs) ──
target_encode_dict = {}
for t in TARGET_COLS:
    te = df_fe.groupby('model_type')[t].mean()
    df_fe[f'{t}_model_mean'] = df_fe['model_type'].map(te)
    target_encode_dict[t] = te.to_dict()

new_cols = sorted(set(df_fe.columns) - set(df_all.columns))
print(f'Engineered {len(new_cols)} features: {", ".join(new_cols)}')


Engineered 19 features: arch_gen, avg_e2e_latency_seconds_model_mean, clock_ratio, comp_latency, compute_proxy, energy_efficiency_tokens_per_joule_model_mean, gpu_power_draw_watts_model_mean, gqa_ratio, hidden_per_layer, is_offline, log_compute, log_params, log_transistor, mem_latency, mem_tier_bw, model_bw_ratio, model_compute_ratio, throughput_tokens_per_second_model_mean, total_tokens


In [4]:
EXCLUDE = set(TARGET_COLS + ['experiment_id'])

feature_cols = [c for c in df_fe.columns
                if c not in EXCLUDE
                and df_fe[c].dtype in [np.float64, np.int64, np.int32, np.float32, 'object', 'bool']]

num_cols = [c for c in feature_cols if df_fe[c].dtype in [np.float64, np.int64, np.int32, np.float32]]
cat_cols = [c for c in feature_cols if df_fe[c].dtype == 'object']

print(f'{len(feature_cols)} features ({len(num_cols)} num + {len(cat_cols)} cat)')
print(f'Cat: {cat_cols}')


33 features (33 num + 0 cat)
Cat: []


In [5]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale', StandardScaler())]), num_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='unknown')),
                      ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat_cols),
])

X = df_fe[feature_cols].copy()
y = df_fe[TARGET_COLS].copy()
mask = y.notna().all(axis=1)
X, y = X[mask], y[mask]
print(f'{len(X)} samples x {len(feature_cols)} features')

555 samples x 33 features


## 2. 5 模型 Stacking CV

| 模型 | 为什么选它 |
|------|----------|
| **LGB** | 表格数据最强泛化，直方图加速 |
| **XGB** | 与 LGB 互补的正则化思路 |
| **RF** | 多样性来源，深度树捕捉高阶交互 |
| **KRR** | RBF 核在物理代理特征（连续+非线性）上表现好 |
| **CatBoost** | 原生 Ordered Boosting，类别特征处理强 |

In [6]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# ── Shared model params ──
lgb_shared = {'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.02,
              'subsample': 0.7, 'colsample_bytree': 0.7,
              'reg_lambda': 3, 'reg_alpha': 0.3,
              'min_child_samples': 15, 'random_state': RANDOM_SEED,
              'n_jobs': -1, 'verbosity': -1}

xgb_shared = {'n_estimators': 500, 'max_depth': 7, 'learning_rate': 0.02,
              'subsample': 0.7, 'colsample_bytree': 0.7,
              'reg_lambda': 3, 'reg_alpha': 0.3,
              'random_state': RANDOM_SEED, 'n_jobs': -1, 'verbosity': 0}

rf_shared  = {'n_estimators': 300, 'max_depth': 15,
              'min_samples_leaf': 3, 'max_features': 0.5,
              'random_state': RANDOM_SEED, 'n_jobs': -1}

krr_shared = {'kernel': 'rbf', 'alpha': 1.0, 'gamma': 0.1}

cb_shared  = {'iterations': 500, 'depth': 7, 'learning_rate': 0.03,
              'l2_leaf_reg': 3, 'random_seed': RANDOM_SEED,
              'verbose': False, 'thread_count': -1}

MODEL_NAMES = ['lgb', 'xgb', 'rf', 'krr', 'cb']
cv_stack = {t: pd.DataFrame(index=X.index, columns=MODEL_NAMES) for t in TARGET_COLS}

for target in TARGET_COLS:
    print(f'\n{"="*60}')
    print(f'{target}')
    print(f'{"="*60}')

    y_target = y[target].values

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y_target[tr_idx], y_target[val_idx]
        X_tr_p = preprocessor.fit_transform(X_tr)
        X_val_p = preprocessor.transform(X_val)

        # LGB
        p = lgb.LGBMRegressor(**lgb_shared).fit(X_tr_p, y_tr).predict(X_val_p)
        cv_stack[target].iloc[val_idx, 0] = p

        # XGB
        p = XGBRegressor(**xgb_shared).fit(X_tr_p, y_tr).predict(X_val_p)
        cv_stack[target].iloc[val_idx, 1] = p

        # RF
        p = RandomForestRegressor(**rf_shared).fit(X_tr_p, y_tr).predict(X_val_p)
        cv_stack[target].iloc[val_idx, 2] = p

        # KRR
        p = KernelRidge(**krr_shared).fit(X_tr_p, y_tr).predict(X_val_p)
        cv_stack[target].iloc[val_idx, 3] = p

        # CatBoost
        p = CatBoostRegressor(**cb_shared).fit(X_tr_p, y_tr).predict(X_val_p)
        cv_stack[target].iloc[val_idx, 4] = p

        # Ensemble: simple average of raw predictions
        stack_vals = cv_stack[target].iloc[val_idx, :].values.astype('float64')
        p_avg = stack_vals.mean(axis=1)
        mae = mean_absolute_error(y_val, p_avg)
        print(f'  Fold {fold+1}: MAE = {mae:.4f}')



gpu_power_draw_watts


  Fold 1: MAE = 10.5470


  Fold 2: MAE = 9.4046


  Fold 3: MAE = 10.5716


  Fold 4: MAE = 10.7205


...

  Fold 1: MAE = 581.7723


  Fold 2: MAE = 326.8973


  Fold 3: MAE = 306.6686


  Fold 4: MAE = 271.4908


  Fold 5: MAE = 221.7049


In [7]:
# ── Ridge meta-learner ──
meta_models = {}
print(f'\n{"="*60}')
print('Ridge Stacking — CV Results')
print(f'{"="*60}')

overall_r2, overall_wmape = [], []

for target in TARGET_COLS:
    y_t = y[target].values
    stack_feat = cv_stack[target].values.astype('float64')
    meta = Ridge(alpha=1.0)
    meta.fit(stack_feat, y_t)
    meta_models[target] = meta

    y_pred = meta.predict(stack_feat)
    r2 = r2_score(y_t, y_pred)
    mae = mean_absolute_error(y_t, y_pred)
    wmape = np.sum(np.abs(y_t - y_pred)) / np.sum(np.abs(y_t)) * 100
    overall_r2.append(r2)
    overall_wmape.append(wmape)

    coef = meta.coef_
    weights = ', '.join([f'{m}={coef[i]:.3f}' for i, m in enumerate(MODEL_NAMES)])
    print(f'\n{target}:')
    print(f'  R^2={r2:.4f}  MAE={mae:.4f}  WMAPE={wmape:.2f}%')
    print(f'  Weights: {weights}')

print(f'\n{"-"*40}')
print(f'CV Avg R^2:   {np.mean(overall_r2):.4f}')
print(f'CV Avg WMAPE: {np.mean(overall_wmape):.2f}%')



Ridge Stacking — CV Results

gpu_power_draw_watts:
  R^2=0.9947  MAE=4.4745  WMAPE=2.47%
  Weights: lgb=0.165, xgb=0.414, rf=-0.140, krr=-0.009, cb=0.583

avg_e2e_latency_seconds:
  R^2=0.8753  MAE=4.2839  WMAPE=19.17%
  Weights: lgb=0.129, xgb=-0.187, rf=0.045, krr=-0.144, cb=1.137

energy_efficiency_tokens_per_joule:
  R^2=0.9882  MAE=1.2081  WMAPE=10.53%
  Weights: lgb=0.121, xgb=0.446, rf=-0.045, krr=-0.292, cb=0.712

throughput_tokens_per_second:
  R^2=0.9906  MAE=272.1290  WMAPE=9.85%
  Weights: lgb=0.246, xgb=-0.315, rf=-0.084, krr=0.078, cb=1.099

----------------------------------------
CV Avg R^2:   0.9622
CV Avg WMAPE: 10.51%


## 3. 全量训练 + 预测

In [8]:
# Train all 5 base models on full data
final_models = {}
X_full = preprocessor.fit_transform(X)

for target in TARGET_COLS:
    y_tr = y[target].values
    print(f'Training: {target}...')
    m_lgb = lgb.LGBMRegressor(**lgb_shared)
    m_lgb.fit(X_full, y_tr)
    m_xgb = XGBRegressor(**xgb_shared)
    m_xgb.fit(X_full, y_tr)
    m_rf = RandomForestRegressor(**rf_shared)
    m_rf.fit(X_full, y_tr)
    m_krr = KernelRidge(**krr_shared)
    m_krr.fit(X_full, y_tr)
    m_cb = CatBoostRegressor(**cb_shared)
    m_cb.fit(X_full, y_tr)
    final_models[target] = {
        'lgb': m_lgb, 'xgb': m_xgb, 'rf': m_rf, 'krr': m_krr, 'cb': m_cb
    }

print('All final models trained.')


Training: gpu_power_draw_watts...


Training: avg_e2e_latency_seconds...


Training: energy_efficiency_tokens_per_joule...


Training: throughput_tokens_per_second...


All final models trained.


In [9]:
def predict_set(df_test, name):
    df_fe = engineer_features(df_test)

    # Apply target encoding from training set
    for t in TARGET_COLS:
        te = target_encode_dict[t]
        df_fe[f'{t}_model_mean'] = df_fe['model_type'].map(te).fillna(
            sum(te.values()) / len(te)  # global mean for unseen model types
        )

    avail = [c for c in feature_cols if c in df_fe.columns]
    X_t = df_fe[avail].copy()
    for col in set(feature_cols) - set(avail):
        X_t[col] = 0.0 if col in num_cols else 'unknown'
    X_t = X_t[feature_cols]
    X_t_p = preprocessor.transform(X_t)

    preds = {}
    for target in TARGET_COLS:
        models = final_models[target]
        meta = meta_models[target]
        stack = np.column_stack([
            models[m].predict(X_t_p) for m in MODEL_NAMES
        ]).astype('float64')
        preds[target] = meta.predict(stack)

    df_pred = pd.DataFrame(preds)
    df_pred.to_csv(f'{name}_predict.csv', index=False)
    print(f'Saved: {name}_predict.csv')
    return df_pred

pred_a = predict_set(df_a, 'A')
pred_b = predict_set(df_b, 'B')


Saved: A_predict.csv


Saved: B_predict.csv


## 4. vs Ground Truth

In [10]:
for label, pdf, gt_path in [('A', pred_a, 'A_ground_truth.csv'),
                             ('B', pred_b, 'B_ground_truth.csv')]:
    gt = pd.read_csv(gt_path)
    print(f'\n{"="*60}')
    print(f'{label} Set vs Ground Truth')
    print(f'{"="*60}')
    r2s, wmapes = [], []
    for t in TARGET_COLS:
        r2 = r2_score(gt[t], pdf[t])
        mae = mean_absolute_error(gt[t], pdf[t])
        wmape = np.sum(np.abs(gt[t] - pdf[t])) / np.sum(np.abs(gt[t])) * 100
        r2s.append(r2); wmapes.append(wmape)
        print(f'  {t}: R^2={r2:.4f}  MAE={mae:.4f}  WMAPE={wmape:.2f}%')
    print(f'  {"-"*40}')
    print(f'  Avg R^2={np.mean(r2s):.4f}  Avg WMAPE={np.mean(wmapes):.2f}%')


A Set vs Ground Truth
  gpu_power_draw_watts: R^2=0.5987  MAE=48.6146  WMAPE=23.01%
  avg_e2e_latency_seconds: R^2=-2.2477  MAE=14.5534  WMAPE=193.38%
  energy_efficiency_tokens_per_joule: R^2=0.9301  MAE=4.3951  WMAPE=29.48%
  throughput_tokens_per_second: R^2=0.5140  MAE=2882.5713  WMAPE=60.30%
  ----------------------------------------
  Avg R^2=-0.0512  Avg WMAPE=76.54%

B Set vs Ground Truth
  gpu_power_draw_watts: R^2=0.5798  MAE=50.9532  WMAPE=23.84%
  avg_e2e_latency_seconds: R^2=-1.6923  MAE=14.6744  WMAPE=176.33%
  energy_efficiency_tokens_per_joule: R^2=0.9223  MAE=4.8694  WMAPE=30.87%
  throughput_tokens_per_second: R^2=0.5451  MAE=2932.8725  WMAPE=58.46%
  ----------------------------------------
  Avg R^2=0.0887  Avg WMAPE=72.38%


## 5. 特征重要性（LGB）

In [11]:
for target in TARGET_COLS:
    if target in final_models:
        imp = pd.DataFrame({
            'feature': final_models[target]['lgb'].feature_name_,
            'importance': final_models[target]['lgb'].feature_importances_
        }).sort_values('importance', ascending=False)
        print(f'\n--- {target} (top 15) ---')
        print(imp.head(15).to_string(index=False))


--- gpu_power_draw_watts (top 15) ---
  feature  importance
 Column_2        2030
 Column_1        1361
Column_25        1297
 Column_3        1285
Column_21        1051
Column_20         898
Column_29         819
 Column_0         664
 Column_9         647
 Column_4         520
 Column_5         466
 Column_6         445
Column_30         413
Column_23         376
Column_24         336

--- avg_e2e_latency_seconds (top 15) ---
  feature  importance
Column_21        2147
 Column_2        1316
Column_30        1082
 Column_3        1044
 Column_4        1003
 Column_1         874
Column_25         719
Column_28         685
 Column_5         583
Column_29         580
Column_20         544
 Column_0         487
Column_31         422
 Column_6         316
Column_32         215

--- energy_efficiency_tokens_per_joule (top 15) ---
  feature  importance
 Column_1        1450
 Column_2        1296
Column_21        1188
Column_20        1188
 Column_4         971
Column_25         678
 Column_